In [1]:
from torch.utils.data import DataLoader, random_split 
from firealarm_net import FireAlarmCNN, MelDataset
from sklearn.preprocessing import LabelEncoder 
import torch.nn as nn
import pandas as pd
import numpy as np 
import torch 
import os

Apple


In [2]:
# Load metadata file
df = pd.read_csv('model/features_metadata.csv')

features, labels = [], []

# Load mel-spectrogram arrays and their labels
for _, row in df.iterrows():
    file_path = row['file']
    label = row['label']
    feature = np.load(os.path.join("features", os.path.basename(file_path)))
    features.append(feature)
    labels.append(label)

features = np.array(features)
labels = np.array(labels)

# Normalize
mean = features.mean()
std = features.std()
features = (features - mean) / (std + 1e-8)

# Add channel dimension (1, n_mels, time)
features = features[:, None, :, :]

In [3]:
# Encode labels
encoder = LabelEncoder()
labels = encoder.fit_transform(labels)

# Build dataset and split (80/20)
dataset = MelDataset(features, labels)
train_len = int(0.8 * len(dataset))
val_len = len(dataset) - train_len
train_ds, val_ds = random_split(dataset, [train_len, val_len])

batch_size = 32
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False, num_workers=0)

# Instantiate model, optimizer, loss and scheduler
model = FireAlarmCNN(num_classes=len(np.unique(labels)))
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [4]:
num_epochs = 30
best_val_acc = 0.0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for Xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(Xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * Xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            outputs = model(Xb)
            loss = criterion(outputs, yb)
            val_loss += loss.item() * Xb.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == yb).sum().item()
            val_total += yb.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{num_epochs}  "
          f"TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"TrainAcc={train_acc:.4f}  ValAcc={val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'model/best_model.pt')


model.load_state_dict(torch.load("model/best_model.pt", map_location="cpu"))
print(f"Best Model Accuracy: {best_val_acc:.4f}")

Epoch 1/30  TrainLoss=0.9580  ValLoss=0.9837  TrainAcc=0.5149  ValAcc=0.5529
Epoch 2/30  TrainLoss=0.7486  ValLoss=0.8787  TrainAcc=0.6935  ValAcc=0.5765
Epoch 3/30  TrainLoss=0.6966  ValLoss=0.8037  TrainAcc=0.6845  ValAcc=0.6824
Epoch 4/30  TrainLoss=0.6310  ValLoss=0.7046  TrainAcc=0.7321  ValAcc=0.7176
Epoch 5/30  TrainLoss=0.5898  ValLoss=0.6562  TrainAcc=0.8006  ValAcc=0.7647
Epoch 6/30  TrainLoss=0.5199  ValLoss=0.6162  TrainAcc=0.8095  ValAcc=0.8471
Epoch 7/30  TrainLoss=0.5097  ValLoss=0.5671  TrainAcc=0.8244  ValAcc=0.8588
Epoch 8/30  TrainLoss=0.5065  ValLoss=0.5613  TrainAcc=0.8065  ValAcc=0.8000
Epoch 9/30  TrainLoss=0.4989  ValLoss=0.5134  TrainAcc=0.8125  ValAcc=0.8353
Epoch 10/30  TrainLoss=0.4506  ValLoss=0.4966  TrainAcc=0.8393  ValAcc=0.7882
Epoch 11/30  TrainLoss=0.4509  ValLoss=0.4330  TrainAcc=0.8333  ValAcc=0.8824
Epoch 12/30  TrainLoss=0.3969  ValLoss=0.5074  TrainAcc=0.8631  ValAcc=0.8353
Epoch 13/30  TrainLoss=0.3580  ValLoss=0.3943  TrainAcc=0.8899  ValAcc=0.